In [2]:
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report
from sklearn.multioutput import MultiOutputClassifier
from sklearn.base import BaseEstimator
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
import sys
import os
import joblib

In [3]:
df = pd.read_csv(r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\data\raw_data\cleaned_raw_data.csv")
df.head(2)

,Name,Range_km,Efficiency,Weight_kg,Acceleration(0-100),Firth_stop_range_km,Battery_kWh,Fast_charger(kW),Towing_kg,Cargo_volume,Price/Range(km),Price_Euro
0,BMW iX3 50 xDrive,610,178,2360,4.9,742,108.7,230,2000,578,114,70.90
1,Tesla Model Y Long Range RWD,490,157,1953,7.2,574,77.0,130,1600,952,100,47.97


In [4]:
df['Range_km_level'] = pd.cut(df['Range_km'],bins=[0, 340, 440, np.inf], labels=[0,1,2])
df['Price_level'] = pd.cut(df['Price_Euro'], bins=[0, 45, 64, np.inf], labels=[0,1,2])


x = df.drop(columns=['Range_km_level', 'Price_level', 'Range_km', 'Price_Euro'])
y = df[['Range_km_level', 'Price_level']].astype(int)

num_cols = x.select_dtypes(include=[np.number]).columns.to_list()

preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', MinMaxScaler(),num_cols),
        ('drop_cols', 'drop', ['Name'])
    ],
    remainder='passthrough' 
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MultiOutputClassifier(XGBClassifier(learning_rate=0.2, random_state=99, max_depth=3, n_estimators=200)))
])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=49)

pipeline.fit(x_train,y_train)

y_pred = pipeline.predict(x_test)


print("\n=== Range_km_level (0: short, 1: middle, 2: long) uchun Model Natijalari ===")
print(classification_report(y_test.iloc[:, 0], y_pred[:, 0]))

print("\n=== Price_level (0: cheap, 1: affordable, 2: expensive) uchun Model Natijalari ===")
print(classification_report(y_test.iloc[:, 1], y_pred[:, 1]))


=== Range_km_level (0: short, 1: middle, 2: long) uchun Model Natijalari ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99        78
           1       0.98      0.94      0.96        88
           2       0.95      0.99      0.97        74

    accuracy                           0.97       240
   macro avg       0.97      0.97      0.97       240
weighted avg       0.97      0.97      0.97       240


=== Price_level (0: cheap, 1: affordable, 2: expensive) uchun Model Natijalari ===
              precision    recall  f1-score   support

           0       0.89      0.91      0.90        70
           1       0.90      0.88      0.89        90
           2       0.96      0.96      0.96        80

    accuracy                           0.92       240
   macro avg       0.92      0.92      0.92       240
weighted avg       0.92      0.92      0.92       240



In [5]:
# modelni saqlash
path = r"C:\Users\bunyo\OneDrive\Desktop\AI_Course\ModularProgramProjects\finalProject\notebooks\pipline"
model_name = "EVcar_model.joblib"
model_path = os.path.join(path, model_name)
joblib.dump(pipeline, model_path)

['C:\\Users\\bunyo\\OneDrive\\Desktop\\AI_Course\\ModularProgramProjects\\finalProject\\notebooks\\pipline\\EVcar_model.joblib']